In [4]:
import cv2
import numpy as np
import os

from deepface import DeepFace
from tensorflow.keras.models import load_model

print("Starting Smart AI System...")

# =========================
# LOAD MODELS
# =========================

emotion_model = load_model("emotion_model.h5", compile=False)
print("Emotion model loaded")

liveness_model = load_model("liveness_model.h5", compile=False)
print("Liveness model loaded")

# =========================
# DATABASE
# =========================

FACE_DB = "dataset/faces_db"

# =========================
# LABELS
# =========================

emotion_labels = [
    'Angry',
    'Disgust',
    'Fear',
    'Happy',
    'Neutral',
    'Sad',
    'Surprise'
]

# =========================
# FACE DETECTOR
# =========================

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    'haarcascade_frontalface_default.xml'
)

print("Face detector loaded")

# =========================
# START WEBCAM
# =========================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Webcam not opened")
else:
    print("Webcam opened")

print("Press Q to quit")

# =========================
# MAIN LOOP
# =========================

while True:

    ret, frame = cap.read()

    if not ret:
        print("Frame not captured")
        break

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:

        try:

            # =========================
            # EMOTION
            # =========================

            emotion_face = gray[
                y:y+h,
                x:x+w
            ]

            emotion_face = cv2.resize(
                emotion_face,
                (48, 48)
            )

            emotion_face = emotion_face / 255.0

            emotion_face = emotion_face.reshape(
                1,
                48,
                48,
                1
            )

            emotion_prediction = emotion_model.predict(
                emotion_face,
                verbose=0
            )

            emotion_index = np.argmax(
                emotion_prediction
            )

            emotion = emotion_labels[
                emotion_index
            ]

            # =========================
            # LIVENESS
            # =========================

            live_face = frame[
                y:y+h,
                x:x+w
            ]

            live_face = cv2.resize(
                live_face,
                (128, 128)
            )

            live_face = live_face / 255.0

            live_face = live_face.reshape(
                1,
                128,
                128,
                3
            )

            live_prediction = liveness_model.predict(
                live_face,
                verbose=0
            )

            confidence = live_prediction[0][0]

            if confidence > 0.5:
                live_status = "Spoof"
            else:
                live_status = "Live"

            # =========================
            # FACE RECOGNITION
            # =========================

            results = DeepFace.find(
                img_path=frame,
                db_path=FACE_DB,
                model_name="Facenet512",
                enforce_detection=False,
                silent=True
            )

            if len(results) > 0 and not results[0].empty:

                identity = results[0].iloc[0][
                    'identity'
                ]

                person_name = os.path.basename(
                    os.path.dirname(identity)
                )

            else:
                person_name = "Unknown"

            # =========================
            # DISPLAY
            # =========================

            display_text = (
                f"{person_name} | "
                f"{emotion} | "
                f"{live_status}"
            )

            cv2.rectangle(
                frame,
                (x, y),
                (x+w, y+h),
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                display_text,
                (x, y-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

        except Exception as e:
            print("Error:", e)

    cv2.imshow(
        "Smart AI Recognition System",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# =========================
# RELEASE
# =========================

cap.release()

cv2.destroyAllWindows()

Starting Smart AI System...


TypeError: Error when deserializing class 'InputLayer' using config={'batch_shape': [None, 48, 48, 1], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer', 'optional': False}.

Exception encountered: Unrecognized keyword arguments: ['batch_shape']